# kwargs-pass-through-recipe — worked example 1: Kwargs flow to both the forward call and the stored Recipe

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kwargs-pass-through-recipe`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

When wrapping a numpy/torch function, keyword arguments like `axis` or `keepdims` must be forwarded in two places: to the actual function call so the computation is correct, and to the `Recipe` dataclass so the backward function can replay the same kwargs at reverse time. Omitting kwargs from the Recipe is a silent bug — the forward result looks right, but the backward will use wrong axis information.

## Worked solution

Step 1: Implement `wrap_forward_fn(fwd_fn)`. Inside `tensor_func(*args, **kwargs)`, unbox MiniTensor inputs: `raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)`.

Step 2: Call `fwd_fn(*raw_args, **kwargs)` to compute the forward result. The kwargs must reach this call — if you call `fwd_fn(*raw_args)` without `**kwargs`, then `sum(x, axis=1)` silently reduces the wrong axis.

Step 3: Build the `Recipe`: `Recipe(fwd_fn, raw_args, kwargs, parents)`. The same `kwargs` dict passed to the wrapper is stored here — not the function's defaults, not an empty dict.

Step 4: Wrap the output and attach the Recipe. Call the wrapper with `axis=1` and verify both the forward result shape and the Recipe's stored kwargs.

In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Any, Callable, Optional

@dataclass
class Recipe:
    func: Any
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array):
        self.array = array
        self.recipe: Optional[Recipe] = None

def wrap_forward_fn(fwd_fn: Callable) -> Callable:
    def tensor_func(*args, **kwargs):
        # Unbox MiniTensor inputs
        raw_args = tuple(a.array if isinstance(a, MiniTensor) else a for a in args)
        # Forward call receives kwargs
        out_raw = fwd_fn(*raw_args, **kwargs)
        # Build Recipe with the same kwargs
        parents = {i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)}
        out = MiniTensor(out_raw)
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func

# Wrap numpy sum (which accepts axis= and keepdims=)
wrapped_sum = wrap_forward_fn(np.sum)

x = MiniTensor(np.arange(12.0).reshape(3, 4))

# Call with axis=1
out = wrapped_sum(x, axis=1)
print('output array:', out.array)                    # [6. 22. 38.]
print('output shape:', out.array.shape)              # (3,)
print('recipe kwargs:', out.recipe.kwargs)           # {'axis': 1}
assert out.recipe.kwargs == {'axis': 1}

# Call with no kwargs
out_all = wrapped_sum(x)
print('sum of all:', out_all.array)                  # 66.0
print('empty kwargs:', out_all.recipe.kwargs)        # {}
assert out_all.recipe.kwargs == {}